# 3.

In [1]:
%pip install python-dotenv --upgrade --quiet langchain langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

In [3]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key:  ········


In [4]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

In [6]:
question = "Roger has 5 shuttles. He buys 2 more cans of shuttles. Each can has 3 shuttles. How many does he have now?"

prompt_standard = f"Answer this question: {question}"
print("--- STANDARD (Llama3.1-8b) ---")
print(llm.invoke(prompt_standard).content)

--- STANDARD (Llama3.1-8b) ---
To find out how many shuttles Roger has now, we need to add the shuttles he already had to the shuttles he bought.

Roger initially had 5 shuttles. 

He bought 2 cans of shuttles, and each can has 3 shuttles. So, he bought 2 * 3 = 6 shuttles.

Now, we add the shuttles he already had to the shuttles he bought: 5 + 6 = 11 shuttles.

So, Roger now has 11 shuttles.


In [7]:
prompt_cot = f"Answer this question. Let's think step by step. {question}"

print("--- Chain of Thought (Llama3.1-8b) ---")
print(llm.invoke(prompt_cot).content)

--- Chain of Thought (Llama3.1-8b) ---
To find out how many shuttles Roger has now, we need to follow these steps:

1. Roger initially has 5 shuttles.
2. He buys 2 more cans of shuttles. Each can has 3 shuttles, so he buys 2 * 3 = 6 shuttles.
3. Now, we need to add the shuttles he bought to the shuttles he already had. So, 5 (initial shuttles) + 6 (new shuttles) = 11 shuttles.

Therefore, Roger now has 11 shuttles.


In [8]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "How can I get a job as a college student?"

prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give me one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'
    
    1: {sol1}
    2: {sol2}
    3: {sol3}
    
    Act as a HR Manager. Pick the most sustainable one (not bribery) and explain why.
    """
)

tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]}) 
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("--- Tree of Thoughts (ToT) Result ---")
print(tot_chain.invoke(problem))

--- Tree of Thoughts (ToT) Result ---
Based on the three proposed solutions, I would recommend **Solution 3: Leverage Your Skills as a Freelancer** as the most sustainable option. Here's why:

1. **Flexibility**: Freelancing allows you to work on your own schedule and choose projects that fit your availability. This flexibility is particularly beneficial for college students who often have varying class schedules and other commitments.
2. **Autonomy**: As a freelancer, you're your own boss, and you have the freedom to decide which projects to take on and how to manage your time. This autonomy can help you develop valuable skills such as time management, self-motivation, and problem-solving.
3. **Networking**: Freelancing allows you to connect with clients and other freelancers in your industry, which can lead to new opportunities and collaborations. This networking can help you build a professional network that can benefit you in the long run.
4. **Scalability**: As a freelancer, you c

In [11]:
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 1-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi") | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance") | llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Horror") | llm | StrOutputParser(),
)

prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three movie ideas for the topic '{topic}':
    1. Sci-Fi: {draft_scifi}
    2. Romance: {draft_romance}
    3. Horror: {draft_horror}
    
    Your task: Create a new Mega-Movie that combines the TECHNOLOGY of Sci-Fi, the PASSION of Romance, and the FEAR of Horror.
    Write one paragraph.
    """
)

got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]}) 
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("--- Graph of Thoughts (GoT) Result ---")
print(got_chain.invoke("Time Travel"))

--- Graph of Thoughts (GoT) Result ---
Here's a thrilling concept that combines the technology of Sci-Fi, the passion of Romance, and the fear of Horror:

"Moonfall" is a heart-pounding, spine-tingling epic that follows Emily, a brilliant and daring physicist who discovers a way to manipulate time using an ancient, mysterious artifact. As she perfects her technology, she becomes obsessed with a chance to relive the moment she first met her long-lost love, James, who vanished under mysterious circumstances years ago. But when Emily travels back in time to relive their first kiss, she unleashes a terrifying force: a malevolent entity from the 13th century, known as "The Shadow," that has been awakened by her meddling with the timeline. As Emily and James fight to survive the horrors unfolding around them, they must navigate a complex web of love, loss, and the consequences of altering their own timeline, all while racing against time to prevent The Shadow from destroying the fabric of re